# Python Append Files

> 📘 **Python Mastery** · Module 06 — File Handling · Lesson 3/4

Lesson 2's `"w"` mode gave you creation — at the price of destruction. This lesson is about the politer modes: `"a"` adds to a file without touching what is already there (the backbone of every log file), `"a+"` lets you read what you just wrote, and `"x"` creates a file *only if it does not exist yet*. We finish by building a real todo logger.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- Use `"a"` mode to extend files while preserving all existing content.
- Remember that append mode does **not** insert `"\n"` — and place line breaks deliberately.
- Build an ever-growing log across many separate `open()` calls.
- Read back appended data with `"a+"`, using `seek(0)` and explaining why it is needed.
- Protect existing files from accidental overwriting with exclusive-create `"x"` mode.
- Choose correctly between `"w"`, `"a"` and `"x"` using a decision table.

## 1. Append Mode: Add Without Wiping

`"a"` opens the file with its cursor parked at the **end**. Everything you write lands after whatever already exists; nothing before it is touched. And like `"w"`, append mode creates the file for free when it is missing — so the same line of code works on day one and on day one-thousand.

**Syntax:**

```python
with open("log.txt", "a", encoding="utf-8") as f:
    f.write("newest event\n")     # joins the end of the story
```

**Example:** seed a small log, then append twice — history stays intact.

In [1]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)
log_path = "sample_data/server.log"

# One-time setup: create the file with a title.
with open(log_path, "w", encoding="utf-8") as f:
    f.write("=== Server Log ===\n")

# Later runs APPEND: nothing above is harmed.
with open(log_path, "a", encoding="utf-8") as f:
    f.write("09:00 server started\n")

with open(log_path, "a", encoding="utf-8") as f:
    f.write("09:05 Sarah logged in\n")

print(Path(log_path).read_text(encoding="utf-8"))

=== Server Log ===
09:00 server started
09:05 Sarah logged in



## 2. Append Does Not Add `"\n"` Either

Same rule as `"w"` (Lesson 2): `.write()` writes exactly the characters you hand it. Appends are even sneakier, though — each new `with` block starts wherever the last one stopped, which is usually *mid-line*. If your entries lack `\n`, the log becomes one long glued sentence.

**Syntax:**

```python
f.write("entry text\n")     # YOU terminate every entry
```

**Example:** first the mistake, then the repair.

In [2]:
path = "sample_data/sticky_notes.txt"

# Two appends, no newlines -> they weld together.
with open(path, "a", encoding="utf-8") as f:
    f.write("buy milk")
with open(path, "a", encoding="utf-8") as f:
    f.write("call the plumber")
print(repr(Path(path).read_text(encoding="utf-8")))

'buy milkcall the plumber'


In [3]:
path = "sample_data/sticky_notes.txt"

# Close off the damaged line, then append properly.
with open(path, "a", encoding="utf-8") as f:
    f.write(" -- oops, forgot the newline!\n")
    f.write("this entry ends cleanly\n")

for line in Path(path).read_text(encoding="utf-8").splitlines():
    print("-", line)

- buy milkcall the plumber -- oops, forgot the newline!
- this entry ends cleanly


## 3. Building a Log Across Multiple Opens

A log's whole point is to survive between program runs. Because append mode always targets the current end of file, three separate `with` blocks — minutes apart, or days apart — weave one continuous timeline. This is exactly how server logs, chat histories, and audit trails grow.

**Syntax:**

```python
def log_event(text):
    with open("events.log", "a", encoding="utf-8") as f:
        f.write(f"{text}\n")      # call it as often as you like
```

**Example:** simulate three study sessions, each its own open-append-close.

In [4]:
from pathlib import Path

sessions = [
    "2026-08-24 | studied variables",
    "2026-08-25 | studied loops",
    "2026-08-26 | studied file handling",
]

# Each block = one 'program run', days apart.
for date_line in sessions:
    with open("sample_data/study_log.txt", "a", encoding="utf-8") as f:
        f.write(date_line + "\n")

history = Path("sample_data/study_log.txt").read_text(encoding="utf-8")
print(history)
print("(Re-running this cell appends AGAIN - logs are honestly cumulative.)")

2026-08-24 | studied variables
2026-08-25 | studied loops
2026-08-26 | studied file handling

(Re-running this cell appends AGAIN - logs are honestly cumulative.)


## 4. Appending Many Entries at Once

Lesson 2's `writelines()` pairs naturally with `"a"`: hand it any iterable of ready-made lines and they all join the end of the file in a single call. Same contract as always — the newlines must already be inside your strings.

**Syntax:**

```python
with open("events.log", "a", encoding="utf-8") as f:
    f.writelines(line + "\n" for line in new_events)
```

**Example:**

In [5]:
from pathlib import Path

batch = [
    "2026-08-26 | imported lesson notes",
    "2026-08-26 | ran every example",
    "2026-08-26 | took a well-earned break",
]

with open("sample_data/study_log.txt", "a", encoding="utf-8") as f:
    f.writelines(entry + "\n" for entry in batch)

print(Path("sample_data/study_log.txt").read_text(encoding="utf-8"))

2026-08-24 | studied variables
2026-08-25 | studied loops
2026-08-26 | studied file handling
2026-08-26 | imported lesson notes
2026-08-26 | ran every example
2026-08-26 | took a well-earned break



## 5. `"a+"`: Write *and* Read Back

Add `+` to any mode and the handle can both write and read: `"r+"`, `"w+"`, `"a+"`. The natural question — *"I just appended, why can't I read it back?"* — leads to today's key concept: the **file cursor**.

> 🔍 **Under the Hood:** a file object tracks one integer: the cursor, your position in the byte stream. Every read or write moves it. `f.tell()` reports where you are; `f.seek(n)` jumps to offset `n` (`0` = start). In `"a"`/`"a+"` mode the cursor starts at the **end** — that is the whole point of append — so a fresh `f.read()` sees only emptiness behind it. Rewind with `f.seek(0)` and the same handle happily reads from the top. One subtlety worth knowing: in append mode *writes* are forced to the end no matter where you seeked — Python protects the "never overwrite" promise.

**Syntax:**

```python
with open("log.txt", "a+", encoding="utf-8") as f:
    f.write("new entry\n")
    f.seek(0)                 # rewind the cursor to the start
    print(f.read())           # now the whole file is visible
```

**Example:**

In [6]:
path = "sample_data/cursor_demo.txt"

# Step 1: the surprise - reading right after appending shows NOTHING.
with open(path, "a+", encoding="utf-8") as f:
    f.write("entry written at position ...\n")
    print("cursor sits at:", f.tell())
    print("read() right now:", repr(f.read()))     # empty! cursor was at the END

cursor sits at: 31
read() right now: ''


In [7]:
path = "sample_data/cursor_demo.txt"

# Step 2: seek(0) rewinds, then the whole file appears.
with open(path, "a+", encoding="utf-8") as f:
    f.write("another entry\n")
    f.seek(0)
    content = f.read()
    print("after seek(0), tell() =", f.tell())
print(content.strip())

after seek(0), tell() = 46
entry written at position ...
another entry


## 6. `"x"`: Exclusive Creation

`"x"` says: *"create this file — but only if it does not exist."* Where `"w"` cheerfully destroys, `"x"` raises `FileExistsError` and keeps the old file safe. Use it whenever overwriting would be a disaster: saving a submission, writing a lock-file so two jobs don't race, exporting a dataset snapshot once and only once.

**Syntax:**

```python
try:
    with open("results.csv", "x", encoding="utf-8") as f:
        f.write(...)
except FileExistsError:
    ...   # refuse politely instead of clobbering
```

**Example:**

In [8]:
from pathlib import Path

submission = Path("sample_data", "exam_submission.txt")

try:
    with open(submission, "x", encoding="utf-8") as f:
        f.write("Sarah's final answers, locked forever.\n")
    print("First save: created.")
except FileExistsError:
    print("First save: already existed -> FileExistsError caught.")

# A second attempt cannot destroy the first.
try:
    with open(submission, "x", encoding="utf-8") as f:
        f.write("blank scribbles\n")
except FileExistsError as e:
    print("Second save refused:", type(e).__name__)

print("Content still intact:", submission.read_text(encoding="utf-8").strip())

First save: created.
Second save refused: FileExistsError
Content still intact: Sarah's final answers, locked forever.


## 7. Choosing the Mode: `"w"` vs `"a"` vs `"x"`

| You want to... | Mode | Old content | Missing file | Existing file |
|----------------|------|-------------|--------------|---------------|
| Replace the file completely | `"w"` | **destroyed** | created | wiped |
| Keep history, add to the end | `"a"` | preserved | created | extended |
| Create only if absent | `"x"` | never touched | created | **error** (`FileExistsError`) |

Memory hook: **w** = wipe, **a** = add, **x** = e**x**clusive.

## 8. Mini-Project: A Todo Logger

Time to combine everything: a tiny but genuinely useful tool. `add_todo()` appends timestamped entries (surviving between runs); `show_todos()` streams the file back line-by-line. Notice how each function opens the file fresh — state lives on disk, not in variables, which is precisely why the list survives after the notebook restarts.

**Example:**

In [9]:
from pathlib import Path
from datetime import datetime

TODO_FILE = Path("sample_data", "todos.txt")

def add_todo(text, when=None):
    """Append one timestamped task. Fixed timestamps keep demos deterministic."""
    stamp = (when or datetime(2026, 8, 26, 12, 0)).strftime("%Y-%m-%d %H:%M")
    with open(TODO_FILE, "a", encoding="utf-8") as f:
        f.write(f"[{stamp}] {text}\n")

def show_todos():
    """Stream the todo file back, newest last."""
    if not TODO_FILE.exists():
        print("(no todos yet - add some!)")
        return
    print(f"--- {TODO_FILE.name} ---")
    with open(TODO_FILE, encoding="utf-8") as f:
        for number, line in enumerate(f, start=1):
            print(f"{number}. {line.rstrip(chr(10))}")

# Simulated input: Sarah plans her afternoon.
add_todo("Finish append-mode lesson", datetime(2026, 8, 26, 14, 0))
add_todo("Do the exercises", datetime(2026, 8, 26, 15, 30))
add_todo("Teach little brother loops", datetime(2026, 8, 26, 18, 0))

show_todos()

--- todos.txt ---
1. [2026-08-26 14:00] Finish append-mode lesson
2. [2026-08-26 15:30] Do the exercises
3. [2026-08-26 18:00] Teach little brother loops


In [10]:
# The logger keeps working across 'runs' - add one more later today.
add_todo("Review CSV lesson notes", datetime(2026, 8, 26, 20, 45))
show_todos()

--- todos.txt ---
1. [2026-08-26 14:00] Finish append-mode lesson
2. [2026-08-26 15:30] Do the exercises
3. [2026-08-26 18:00] Teach little brother loops
4. [2026-08-26 20:45] Review CSV lesson notes


One more function completes the tool: **finishing** a task. There is no "delete one line" mode — instead we read every line, drop the completed one, and *rewrite* the file with `"w"`. Notice the natural division of labour that emerged: `"a"` grows the file, `"w"` reshapes it.

In [11]:
def complete_todo(number):
    """Remove task #number by rewriting the file without it."""
    with open(TODO_FILE, encoding="utf-8") as f:
        lines = f.readlines()
    try:
        removed = lines.pop(number - 1)
    except IndexError:
        print(f"No task number {number}.")
        return
    with open(TODO_FILE, "w", encoding="utf-8") as f:   # reshape with "w"
        f.writelines(lines)
    print("Completed:", removed.strip())

complete_todo(2)
show_todos()

Completed: [2026-08-26 15:30] Do the exercises
--- todos.txt ---
1. [2026-08-26 14:00] Finish append-mode lesson
2. [2026-08-26 18:00] Teach little brother loops
3. [2026-08-26 20:45] Review CSV lesson notes


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---------|---------|-----|
| Appending without `"\n"` | Entries weld into one endless line | End every entry with `"\n"` yourself |
| Using `"a"` then expecting `.read()` to work | `io.UnsupportedOperation` — plain `"a"` cannot read | Use `"a+"` and `seek(0)` first |
| Forgetting `seek(0)` in `"a+"` | `read()` returns `""` and you think the file is empty | `f.seek(0)` before reading |
| Re-running a demo script repeatedly | Log lines duplicate — append is honest about reruns | Accept it (real logging!) or use `"w"` for resettable outputs |
| Choosing `"w"` for anything historical | Yesterday's data gone without warning | `"a"` for histories, `"x"` for protected snapshots |

## 💡 Best Practices & Pro Tips

- Reach for `"a"` the moment a file is a *history*: logs, metrics, messages, high-score tables.
- Timestamp every log line (`datetime.now().strftime(...)`) — future-you debugging at midnight will be grateful.
- Prefer `"x"` over `"w"` for one-time artefacts (exports, submissions, lock-files): the failure is loud, which beats silent data loss.
- Wrap the append logic in a small function (`log_event`) so formatting lives in exactly one place.
- **AI-engineering relevance:** append-only files power modern ML infrastructure. Training metrics stream to JSONL logs, prompt/response pairs accumulate into fine-tuning datasets, and experiment trackers append one line per run. JSONL itself — one JSON object per line — exists precisely because append + newline-per-record makes huge datasets safe to grow and stream.

## 📌 Summary

| Tool | What it does | Example |
|------|--------------|---------|
| `open(p, "a", encoding=...)` | Open at the end; create if missing | `open("log.txt", "a", encoding="utf-8")` |
| `open(p, "a+")` | Append **and** read | `open("log.txt", "a+")` |
| `f.seek(0)` | Move the cursor to the start | `f.seek(0); text = f.read()` |
| `f.tell()` | Report the cursor's byte offset | `print(f.tell())` |
| `open(p, "x")` | Create exclusively; error if present | `open("snap.csv", "x")` → `FileExistsError` |
| `"\n"` | Still your job, in every write mode | `f.write(entry + "\n")` |

Key takeaways:

- `"a"` preserves history; `"w"` rewrites it; `"x"` refuses to risk it.
- The file cursor explains read-after-write behaviour — `tell()`, `seek()`, and why `"a"` reads feel empty.
- Logs grow across separate opens because append always targets the end of file.
- A four-line helper around `open(..., "a")` is all a real todo/logger tool needs.

## 🔗 Next Lesson

Text files are perfect for sentences — terrible for tabular data. For rows and columns there is a standard format with standard tools: [`../04_CSV/notes.ipynb`](../04_CSV/notes.ipynb).